# Create longitudinal `frontier_labs` tables

Stands up the two **append-only** history tables and seeds today's first snapshot from the current tables.
- `model_benchmarks_history` — a dated copy of `model_benchmarks` each run (every 2 weeks) → track performance & pricing change over time.
- `hiring_snapshots` — a dated copy of each job's tags + status + location each run → reconstruct hiring at any past date.

Neither table is ever overwritten — jobs only **append** a new dated batch.

In [ ]:
from pyspark.sql import functions as F

CATALOG = "fso_market_intelligence"
SCHEMA  = "frontier_labs"

## 1. `model_benchmarks_history` (append-only)
Same columns as `model_benchmarks` + `captured_at` (the snapshot date).

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.model_benchmarks_history (
  id STRING, name STRING, slug STRING, org STRING, country STRING,
  release_date DATE, open_weight BOOLEAN, param_count BIGINT, license STRING,
  modalities ARRAY<STRING>,
  intelligence_index DOUBLE, coding_index DOUBLE, math_index DOUBLE,
  gpqa DOUBLE, hle DOUBLE, mmlu_pro DOUBLE, livecodebench DOUBLE,
  ifbench DOUBLE, lcr DOUBLE, aime_25 DOUBLE,
  price_input DOUBLE, price_output DOUBLE, price_blended DOUBLE,
  tokens_per_sec DOUBLE, ttft DOUBLE,
  captured_at DATE
) USING DELTA
""")
print("created (or already existed): model_benchmarks_history")

## 2. `hiring_snapshots` (append-only)
One dated row per job: tags + status + location. Lean by design — `title`/`url`/`description` stay in `hiring_jobs` (static per job, joinable by `job_id`).

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.hiring_snapshots (
  job_id STRING, company STRING, location STRING, category STRING, sub_area STRING,
  theme STRING, vertical STRING, social_impact BOOLEAN, active BOOLEAN,
  snapshot_date DATE
) USING DELTA
""")
print("created (or already existed): hiring_snapshots")

## 3. Seed today's first snapshot (idempotent — skips if today already loaded)

In [ ]:
# model_benchmarks_history
hist = f"{CATALOG}.{SCHEMA}.model_benchmarks_history"
if spark.table(hist).where(F.col("captured_at") == F.current_date()).limit(1).count() == 0:
    (spark.table(f"{CATALOG}.{SCHEMA}.model_benchmarks")
        .withColumn("captured_at", F.current_date())
        .write.mode("append").saveAsTable(hist))
    print("seeded model_benchmarks_history for today")
else:
    print("today already in model_benchmarks_history — skipped")
print("  history rows:", spark.table(hist).count())

# hiring_snapshots
snap = f"{CATALOG}.{SCHEMA}.hiring_snapshots"
if spark.table(snap).where(F.col("snapshot_date") == F.current_date()).limit(1).count() == 0:
    (spark.table(f"{CATALOG}.{SCHEMA}.hiring_jobs")
        .select("job_id", "company", "location", "category", "sub_area", "theme", "vertical", "social_impact", "active")
        .withColumn("snapshot_date", F.current_date())
        .write.mode("append").saveAsTable(snap))
    print("seeded hiring_snapshots for today")
else:
    print("today already in hiring_snapshots — skipped")
print("  snapshot rows:", spark.table(snap).count())

## Going forward
The scheduled jobs **append** to these (never overwrite):
```python
# benchmarks job, every 2 weeks:
(new_models.withColumn("captured_at", F.current_date())
   .write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.model_benchmarks_history"))

# hiring job, daily:
(current_jobs.select("job_id","company","location","category","sub_area","theme","vertical","social_impact","active")
   .withColumn("snapshot_date", F.current_date())
   .write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.hiring_snapshots"))
```
Change over time then comes from window functions, e.g. `LAG(price_blended) OVER (PARTITION BY id ORDER BY captured_at)`.